# TBD Phase 2 26L: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing libraries/engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: single-node processing speed, parallel execution, memory usage, and result materialization cost.
- **Scalability**: how performance changes with the number of local threads/cores and with Spark executors on a cluster.
- **Physical layout**: how file format, Parquet layout, row groups, sorting, partitioning, and pruning affect IO.
- **Computing models**: in-memory vs. out-of-core processing, SQL vs. DataFrame APIs, eager vs. lazy execution, and streaming execution vs. streaming output.

This notebook is an assignment template. It gives you a common structure and helper code, but you must design your own dataset variant, queries, benchmark implementation, and analysis.


## Submission identity

Before starting the assignment, copy this notebook into your fork of the workshop repository and work on that copy.

Fill in the first code cell with:

- your group number,
- a link to this notebook in your forked GitHub repository,
- names or IDs of group members if required by the instructor.

The submitted notebook should be reachable from your fork. Do not submit a notebook that only exists locally.

In [1]:
# TODO: Fill this in before submitting.
GROUP_ID = 4
NOTEBOOK_URL = "https://github.com/matiuszaa/tbd-workshop-1/blob/master/notebooks/tbd_phase_2_26L.ipynb"
GROUP_MEMBERS = [
    "Mateusz Baran / 304483",
    "Mikołaj Garbowski",
    "Jakub Lemański",
]

assert GROUP_ID is not None, "Set GROUP_ID before running the notebook"
assert "<your-github-user-or-org>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"

## Library/engine capabilities

Use this table as a reference when interpreting your results.

| Library/engine | Query optimizer | Distributed | Arrow-backed | Out-of-core | Parallel local execution | Main APIs |
|---|---|---|---|---|---|---|
| **Pandas 3.0** | no | no | default IO returns NumPy-backed data; `dtype_backend="pyarrow"` returns PyArrow-backed nullable dtypes | no | limited | DataFrame, `pd.col` for selected expression-style usage |
| **Polars** | yes | single-node locally; distributed engine is available in Polars Cloud and is outside this local benchmark | yes | yes | yes | DataFrame, lazy expressions, SQL subset |
| **DuckDB** | yes | no | yes | yes | yes | SQL, relational API |
| **PySpark** | yes | yes | yes, for selected IO/UDF paths | yes | yes | SQL, DataFrame |

The goal is not to prove that one library is always best. The goal is to identify which library/engine is appropriate for a given data size, query shape, memory limit, physical layout, and deployment model.

Use pandas 3.0 in this lab. Two pandas 3.0 behaviours matter for the benchmark: string columns are no longer inferred as generic `object` dtype by default, and Copy-on-Write is the only mutation model. In addition, compare two Pandas Parquet-reading variants where possible:

- default Pandas/NumPy-backed DataFrame: `pd.read_parquet(path)`,
- PyArrow-backed DataFrame: `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`.

Record the pandas version and dtypes in your report.


## Prerequisites

Install the required libraries in your notebook environment. If the course image already contains them, this command should be quick. Pandas 3.0 requires Python 3.11 or newer.

Use current Polars API in new code. In particular, use `collect(engine="streaming")` for streaming execution and use sink methods when you want to write streaming output to disk.

For Pandas, benchmark both the default backend and the PyArrow dtype backend for Parquet reads. The PyArrow-backed variant is especially relevant for string-heavy datasets.


In [4]:
%pip install -U "pandas>=3.0,<3.1" polars duckdb pyspark faker deltalake memory_profiler pyarrow psutil matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 33.5 MB/s eta 0:00:00
   ━━━━━

In [5]:
import gc
import os
import time
import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import psutil
from faker import Faker
from memory_profiler import memory_usage
from pyspark.sql import SparkSession

print("Python:", platform.python_version())
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11+ because it uses pandas 3.0.")
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
if tuple(map(int, pd.__version__.split(".")[:2])) < (3, 0):
    raise RuntimeError("Install pandas 3.0+ before running the benchmark.")
print("DuckDB:", duckdb.__version__)
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))


Python: 3.12.13
Polars: 1.40.1
Pandas: 3.0.3
DuckDB: 1.5.2
CPU logical cores: 2
RAM GiB: 12.67


## Part 1: Data generation with group variants

Each group works with one assigned synthetic data profile. Use your group number to select the variant card below.

Your dataset does not need to match other groups exactly, but it must satisfy the common schema and benchmarking requirements described in this notebook.

Every group must document:
- dataset profile,
- main benchmark row count, plus any additional stress-test row counts if used,
- physical layout and file format choices,
- library versions,
- query intent,
- benchmark results,
- conclusions.

You may use the helper functions below, but you must adapt the dataset to your assigned variant.


## Variant cards for 16 groups

Choose or assign one variant per group.

| Group | Data profile | Required data feature | Suggested query stress |
|---:|---|---|---|
| 1 | Social media posts | tags or hashtags | explode/list handling, top-k |
| 2 | E-commerce orders | products and order values | join, category aggregation |
| 3 | IoT telemetry | device time series | time filters, rolling/window logic |
| 4 | Application logs | status codes and endpoints | selective filters, string columns |
| 5 | Advertising clicks | campaign skew | CTR, skewed group-by, join |
| 6 | Game events | player sessions | high-cardinality group-by |
| 7 | Streaming platform events | watch duration | device/country aggregation |
| 8 | Public transport events | route delays | time and location aggregation |
| 9 | Banking-like transactions | risk/fraud flags | selective filters, top-k, sorting |
| 10 | Web analytics | referrers and pages | funnel-like aggregation |
| 11 | Delivery/logistics events | late status updates | late events, time windows |
| 12 | Education platform activity | courses and students | joins and progress metrics |
| 13 | Weather measurements | missing values | resampling and null handling |
| 14 | Marketplace listings | prices and categories | quantiles, category statistics |
| 15 | Security events | rare alerts | selective filters and high skew |
| 16 | Support tickets | priority and SLA | time-to-resolution metrics |

You may rename columns and categories to fit the chosen profile. Keep enough common structure to run the same engine comparisons.

In [6]:
DOMAIN_CARDS = {
    1: {"name": "Social media posts", "feature": "tags", "stress": "explode/list handling and top-k"},
    2: {"name": "E-commerce orders", "feature": "products", "stress": "joins and category aggregation"},
    3: {"name": "IoT telemetry", "feature": "device time series", "stress": "time filters and rolling/window logic"},
    4: {"name": "Application logs", "feature": "status codes", "stress": "selective filters and string columns"},
    5: {"name": "Advertising clicks", "feature": "campaign skew", "stress": "CTR, skewed group-by, and joins"},
    6: {"name": "Game events", "feature": "player sessions", "stress": "high-cardinality group-by"},
    7: {"name": "Streaming platform events", "feature": "watch duration", "stress": "device/country aggregation"},
    8: {"name": "Public transport events", "feature": "route delays", "stress": "time and location aggregation"},
    9: {"name": "Banking-like transactions", "feature": "risk flags", "stress": "selective filters, top-k, and sorting"},
    10: {"name": "Web analytics", "feature": "referrers", "stress": "funnel-like aggregation"},
    11: {"name": "Delivery/logistics events", "feature": "late status updates", "stress": "late events and time windows"},
    12: {"name": "Education platform activity", "feature": "courses", "stress": "joins and progress metrics"},
    13: {"name": "Weather measurements", "feature": "missing values", "stress": "resampling and null handling"},
    14: {"name": "Marketplace listings", "feature": "prices", "stress": "quantiles and category statistics"},
    15: {"name": "Security events", "feature": "rare alerts", "stress": "selective filters and high skew"},
    16: {"name": "Support tickets", "feature": "priority and SLA", "stress": "time-to-resolution metrics"},
}

assert 1 <= GROUP_ID <= 16, "GROUP_ID must be between 1 and 16"
CARD = DOMAIN_CARDS[GROUP_ID]
CARD

{'name': 'Application logs',
 'feature': 'status codes',
 'stress': 'selective filters and string columns'}

## Dataset requirements

Your generated dataset must contain at least:

- one timestamp column,
- one high-cardinality identifier, such as user, device, session, order, ticket, or transaction id,
- at least two categorical columns,
- at least two numeric metric columns,
- one feature specific to your variant card,
- enough rows to make local benchmark differences visible,
- a Parquet output file or directory.

Recommended starting sizes:

| Scale | Rows | Use case |
|---|---:|---|
| debug | 200,000 | Validate code quickly |
| small | 2,000,000 | Local development and first benchmark |
| medium | 10,000,000 to 20,000,000 | Main benchmark |
| large | 50,000,000+ | Optional stress test |

Use `debug` only while developing. The rendered notebook should report one main benchmark size (`N_ROWS`). If you run additional sizes, put those results in a separate stress-test table and do not mix them with the main benchmark table.

It is acceptable for different groups to generate different random data. Choose one main dataset size for the benchmark and record it as `N_ROWS`. You may use smaller debug data while developing and optional larger data for stress tests, but those extra sizes should be reported separately.

In [7]:
# TODO: Choose the main dataset scale for your final benchmark and verify output paths before generation.
# N_ROWS is the main row count reported for this notebook. Extra row counts are optional stress tests.
# Dataset configuration
SCALE = "small"
SCALE_ROWS = {
    "debug": 200_000,
    "small": 2_000_000,
    "medium": 10_000_000,
    "large": 50_000_000,
}

N_ROWS = SCALE_ROWS[SCALE]
OUTPUT_DIR = Path("../data/phase2_26L") / f"group_{GROUP_ID:02d}"
EVENTS_PATH = OUTPUT_DIR / "events.parquet"
PARTITIONED_EVENTS_DIR = OUTPUT_DIR / "events_partitioned"
OPTIMIZED_EVENTS_PATH = OUTPUT_DIR / "events_optimized.parquet"
DIMENSION_PATH = OUTPUT_DIR / "dimension.parquet"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

# Required negative baseline paths for the file-format/layout task. Do not commit these generated files.
CSV_EVENTS_PATH = OUTPUT_DIR / "events.csv"
JSON_EVENTS_PATH = OUTPUT_DIR / "events.jsonl"

# Leave SEED as None if you want independent data on each generation.
# If you need to reproduce exactly the same dataset later, set SEED to the value stored in the manifest.
SEED = None
RUN_SEED = int(np.random.SeedSequence().entropy) if SEED is None else int(SEED)
rng = np.random.default_rng(RUN_SEED)
fake = Faker()

print("Group:", GROUP_ID, CARD)
print("Rows:", N_ROWS)
print("Run seed recorded in manifest:", RUN_SEED)
print("Output directory:", OUTPUT_DIR)


Group: 4 {'name': 'Application logs', 'feature': 'status codes', 'stress': 'selective filters and string columns'}
Rows: 2000000
Run seed recorded in manifest: 339324030508111664212359641261695187570
Output directory: ../data/phase2_26L/group_04


## Generator template

The helper below creates a common base event table. You should extend it for your variant.

Do not spend most of the assignment writing a perfect data generator. The generator only needs to create data that is large enough and structurally interesting enough for your benchmark questions.

In [8]:
# TODO: Adapt customize_for_variant(...) and generate_dimension_table(...) to your variant.
def skewed_ids(rng, n, max_id, hot_fraction=0.02, hot_probability=0.50):
    hot_count = max(1, int(max_id * hot_fraction))
    ids = rng.integers(hot_count + 1, max_id + 1, size=n)
    hot_mask = rng.random(n) < hot_probability
    ids[hot_mask] = rng.integers(1, hot_count + 1, size=hot_mask.sum())
    return ids


def random_tag_lists(rng, n, vocabulary=None, min_tags=1, max_tags=3):
    vocabulary = np.array(vocabulary or ["ai", "cloud", "spark", "polars", "duckdb", "sql", "etl", "security", "mlops"])
    counts = rng.integers(min_tags, max_tags + 1, size=n)
    tag_ids = rng.integers(0, len(vocabulary), size=(n, max_tags))
    return [[str(vocabulary[tag_ids[i, j]]) for j in range(counts[i])] for i in range(n)]


def generate_base_events(n, rng):
    start = np.datetime64("2026-01-01T00:00:00", "s")
    end = np.datetime64("2026-04-01T00:00:00", "s")
    seconds = int((end - start) / np.timedelta64(1, "s"))
    event_ts = (start + rng.integers(0, seconds, size=n).astype("timedelta64[s]")).astype("datetime64[us]")

    df = pl.DataFrame(
        {
            "event_id": np.arange(1, n + 1),
            "entity_id": skewed_ids(rng, n, max_id=200_000),
            "event_ts": event_ts,
            "category": rng.choice(["A", "B", "C", "D", "E", "F"], size=n),
            "country": rng.choice(["PL", "DE", "FR", "UK", "US", "IN", "BR"], size=n),
            "device": rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.65, 0.25, 0.10]),
            "metric_1": rng.lognormal(mean=4.0, sigma=1.0, size=n).round(3),
            "metric_2": rng.integers(0, 10_000, size=n),
            "tags": random_tag_lists(rng, n),
        }
    )
    return df.with_columns(pl.col("event_ts").dt.date().alias("event_date"))


def customize_for_variant(df, card, rng):
    n = df.height

    endpoints = ["/api/users", "/api/products", "/api/cart", "/api/checkout", "/api/search", "/api/auth"]
    methods = ["GET", "POST", "PUT", "DELETE"]

    status_codes = rng.choice(
        [200, 201, 400, 401, 403, 404, 500, 502, 503],
        size=n,
        p=[0.65, 0.05, 0.05, 0.05, 0.02, 0.15, 0.01, 0.01, 0.01]
    )

    level_map = {"A": "INFO", "B": "INFO", "C": "DEBUG", "D": "WARN", "E": "ERROR", "F": "FATAL"}

    df_custom = df.rename({
        "entity_id": "session_id",
        "category": "log_level",
    }).with_columns([
        pl.Series("status_code", status_codes, dtype=pl.Int32),
        pl.Series("endpoint", rng.choice(endpoints, size=n)),
        pl.Series("http_method", rng.choice(methods, size=n, p=[0.8, 0.15, 0.03, 0.02])),

        (pl.col("metric_1") * 25).cast(pl.Float32).alias("response_time_ms"),
        (pl.col("metric_2") * 10).cast(pl.Int32).alias("payload_size_bytes"),

        pl.col("log_level").replace(level_map),
        pl.when(pl.col("event_id") % 20 == 0).then(None).otherwise(pl.col("device")).alias("device")
    ]).drop(["metric_1", "metric_2"])

    return df_custom


def generate_dimension_table(card, rng):
    endpoints = ["/api/users", "/api/products", "/api/cart", "/api/checkout", "/api/search", "/api/auth"]

    return pl.DataFrame({
        "endpoint": endpoints,
        "microservice": ["user-service", "catalog-service", "order-service", "payment-service", "search-service", "auth-service"],
        "team_owner": ["team-alpha", "team-beta", "team-gamma", "team-gamma", "team-beta", "team-alpha"],
        "sla_timeout_ms": [200, 300, 150, 500, 400, 100]
    })

In [12]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Generowanie zdarzeń (events)...")
base_events = generate_base_events(N_ROWS, rng)
events = customize_for_variant(base_events, CARD, rng)

print("Generowanie wymiarów (dimension)...")
dimension = generate_dimension_table(CARD, rng)

print("Zapisywanie plików Parquet...")
events.write_parquet(EVENTS_PATH, compression="zstd")
dimension.write_parquet(DIMENSION_PATH, compression="zstd")

# Opcjonalny zapis z podziałem na partycje po dacie (przydatne do testów)
events.write_parquet(PARTITIONED_EVENTS_DIR, partition_by="event_date", compression="zstd")

# --- ZOPTYMALIZOWANY LAYOUT (Rozwiązanie TODO) ---
# Optymalizujemy plik pod nasze Zapytanie 1 (szukanie rzadkich status_code i grupowanie po endpoint).
# Sortujemy plik. Dzięki temu wszystkie kody np. 500 będą fizycznie obok siebie.
# Silniki analityczne będą mogły zastosować tzw. "Row Group Pruning" i pominąć 99% pliku z kodami 200.
print("Zapisywanie zoptymalizowanego pliku Parquet...")
events.sort(["status_code", "endpoint"]).write_parquet(
    OPTIMIZED_EVENTS_PATH,
    compression="zstd",
    row_group_size=100_000,
)

# Tworzenie manifestu z logami ze środowiska i metadata
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "group_id": GROUP_ID,
    "variant": CARD,
    "scale": SCALE,
    "rows": int(events.height),
    "run_seed": RUN_SEED,
    "paths": {
        "events": str(EVENTS_PATH),
        "events_partitioned": str(PARTITIONED_EVENTS_DIR),
        "events_optimized": str(OPTIMIZED_EVENTS_PATH),
        "dimension": str(DIMENSION_PATH),
    },
    "environment": {
        "python": platform.python_version(),
        "polars": pl.__version__,
        "pandas": pd.__version__,
        "duckdb": duckdb.__version__,
        "cpu_logical_cores": psutil.cpu_count(logical=True),
        "ram_gib": round(psutil.virtual_memory().total / 2**30, 2),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\n--- Manifest ---")
print(json.dumps(manifest, indent=2))
print("\nWszystkie dane wygenerowane pomyślnie!")

Generowanie zdarzeń (events)...
Generowanie wymiarów (dimension)...
Zapisywanie plików Parquet...
Zapisywanie zoptymalizowanego pliku Parquet...

--- Manifest ---
{
  "created_at_utc": "2026-05-19T18:33:46.864269+00:00",
  "group_id": 4,
  "variant": {
    "name": "Application logs",
    "feature": "status codes",
    "stress": "selective filters and string columns"
  },
  "scale": "small",
  "rows": 2000000,
  "run_seed": 339324030508111664212359641261695187570,
  "paths": {
    "events": "../data/phase2_26L/group_04/events.parquet",
    "events_partitioned": "../data/phase2_26L/group_04/events_partitioned",
    "events_optimized": "../data/phase2_26L/group_04/events_optimized.parquet",
    "dimension": "../data/phase2_26L/group_04/dimension.parquet"
  },
  "environment": {
    "python": "3.12.13",
    "polars": "1.40.1",
    "pandas": "3.0.3",
    "duckdb": "1.5.2",
    "cpu_logical_cores": 2,
    "ram_gib": 12.67
  }
}

✅ Wszystkie dane wygenerowane pomyślnie!


## Dataset sanity checks

Before benchmarking, inspect your schema and basic statistics. Your report should briefly explain why your dataset is suitable for the queries you chose.

In [11]:
# TODO: Inspect schema, row count, null counts, and basic category distributions.
# Keep this section short, but include enough evidence that your data was generated correctly.

print(f"Events table row count: {events.height:,}")
print(f"Dimension table row count: {dimension.height:,}\n")

print("--- Schema of the 'events' table ---")
for col_name, dtype in events.schema.items():
    print(f"{col_name}: {dtype}")

print("\n--- Missing values (Null counts) ---")
display(events.null_count())

print("\n--- HTTP codes distribution (status_code) ---")
display(events.group_by("status_code").len().sort("status_code"))

print("\n--- Log levels distribution (log_level) ---")
display(events.group_by("log_level").len().sort("len", descending=True))

print("\n--- Data sample (first 3 rows) ---")
display(events.head(3))

print("\n--- Dimension table (Microservices directory) ---")
display(dimension)

Events table row count: 2,000,000
Dimension table row count: 6

--- Schema of the 'events' table ---
event_id: Int64
session_id: Int64
event_ts: Datetime(time_unit='us', time_zone=None)
log_level: String
country: String
device: String
tags: List(String)
event_date: Date
status_code: Int32
endpoint: String
http_method: String
response_time_ms: Float32
payload_size_bytes: Int32

--- Missing values (Null counts) ---


event_id,session_id,event_ts,log_level,country,device,tags,event_date,status_code,endpoint,http_method,response_time_ms,payload_size_bytes
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,100000,0,0,0,0,0,0,0



--- HTTP codes distribution (status_code) ---


status_code,len
i32,u32
200,1299848
201,99605
400,100436
401,99792
403,39913
404,300316
500,20176
502,20026
503,19888



--- Log levels distribution (log_level) ---


log_level,len
str,u32
"""INFO""",666382
"""ERROR""",333998
"""WARN""",333611
"""FATAL""",333352
"""DEBUG""",332657



--- Data sample (first 3 rows) ---


event_id,session_id,event_ts,log_level,country,device,tags,event_date,status_code,endpoint,http_method,response_time_ms,payload_size_bytes
i64,i64,datetime[μs],str,str,str,list[str],date,i32,str,str,f32,i32
1,113988,2026-01-24 08:33:08,"""ERROR""","""DE""","""mobile""","[""duckdb"", ""polars""]",2026-01-24,401,"""/api/search""","""GET""",814.25,16850
2,193335,2026-03-03 04:43:28,"""INFO""","""FR""","""mobile""","[""sql"", ""duckdb""]",2026-03-03,200,"""/api/products""","""GET""",2377.524902,21850
3,145269,2026-01-01 17:45:12,"""ERROR""","""FR""","""desktop""","[""cloud""]",2026-01-01,200,"""/api/users""","""GET""",1626.25,63220



--- Dimension table (Microservices directory) ---


endpoint,microservice,team_owner,sla_timeout_ms
str,str,str,i64
"""/api/users""","""user-service""","""team-alpha""",200
"""/api/products""","""catalog-service""","""team-beta""",300
"""/api/cart""","""order-service""","""team-gamma""",150
"""/api/checkout""","""payment-service""","""team-gamma""",500
"""/api/search""","""search-service""","""team-beta""",400
"""/api/auth""","""auth-service""","""team-alpha""",100


This dataset perfectly aligns with the "Application logs" domain profile and provides excellent ground for performance benchmarking:

High Selectivity & Data Skew: The status_code column is intentionally skewed (e.g., millions of 200/INFO logs, but very rare 50x errors). This is ideal for testing the engines' abilities to perform selective filtering and predicate pushdown.

String Operations: We have multiple high-cardinality text columns (endpoint, http_method, log_level). This will expose the performance differences between the Pandas default object backend, the PyArrow string backend, and DuckDB's native string handling.

Join Viability: The presence of the endpoint column in both the massive events table and the small dimension table allows us to benchmark dimension table joins (e.g., aggregating error metrics by team_owner).

## Part 2: Measuring performance

You must use one consistent benchmark protocol for all libraries/engines.

Minimum requirements:

1. Run every benchmark at least three times. Five repetitions are recommended.
2. Run `gc.collect()` before each measured repetition to reduce noise from previous Python allocations.
3. Report median runtime, not only one measurement.
4. Record peak memory where possible.
5. Check that results are logically equivalent across libraries/engines.
6. Store your results in a table.
7. Describe any library/engine-specific settings, such as Pandas dtype backend, thread count, Spark local mode, or DuckDB threads.

**Important for memory benchmarks**: notebook kernels keep allocations and library state between cells. Peak-RSS comparisons are often misleading when all variants run in the same process. For Task 3.1 and any memory-sensitive comparison, prefer running each variant in a fresh process or a small standalone script. If you cannot do that, clearly state this limitation.

You may use the helper shape below, but you need to implement the actual benchmark functions.


In [17]:
# TODO: Implement or adapt benchmark helpers before collecting final results.
BENCHMARK_COLUMNS = [
    "library_engine",
    "mode",
    "query_name",
    "data_format",
    "layout",
    "rows",
    "median_time_s",
    "peak_memory_mb",
    "input_size_mb",
    "result_check",
    "notes",
]

benchmark_results = []
reference_results = {} # Słownik do sprawdzania zgodności wyników między silnikami

def run_benchmark(library_engine, mode, query_name, query_func, notes="", data_format="parquet", layout="default", rows=N_ROWS, iterations=5):
    """
    Uruchamia zapytanie `iterations` razy. Mierzy medianę czasu oraz przyrost pamięci RAM.
    Weryfikuje również logiczną spójność wyników (result_check).
    """
    times = []
    mem_usages = []
    process = psutil.Process(os.getpid())

    for _ in range(iterations):
        gc.collect() # Czyszczenie pamięci przed pomiarem
        mem_before = process.memory_info().rss / (1024 * 1024)

        start_time = time.perf_counter()
        result = query_func()
        end_time = time.perf_counter()

        mem_after = process.memory_info().rss / (1024 * 1024)

        times.append(end_time - start_time)
        mem_usages.append(max(0, mem_after - mem_before))

    median_time = np.median(times)
    avg_peak_mem = np.mean(mem_usages)

    # Sprawdzanie spójności (czy zapytania zwracają tyle samo wierszy w każdym silniku)
    try:
        res_len = result.shape[0] if hasattr(result, "shape") else len(result)
    except:
        res_len = "N/A"

    result_status = "Success"
    if res_len != "N/A":
        if query_name not in reference_results:
            reference_results[query_name] = res_len
            result_status = f"Reference (len: {res_len})"
        elif reference_results[query_name] != res_len:
            result_status = f"Mismatch! (Expected {reference_results[query_name]}, got {res_len})"

    # Dodanie wyników do głównej tabeli
    benchmark_results.append({
        "library_engine": library_engine,
        "mode": mode,
        "query_name": query_name,
        "data_format": data_format,
        "layout": layout,
        "rows": rows,
        "median_time_s": round(median_time, 4),
        "peak_memory_mb": round(avg_peak_mem, 2),
        "input_size_mb": None,
        "result_check": result_status,
        "notes": notes
    })

    print(f"{library_engine[:8]:8} | {mode[:9]:9} | Czas: {median_time:.4f} s | Pamięć: ~{avg_peak_mem:7.2f} MB | {result_status}")

    return result

## Part 3: Student tasks

### Task 1: Design three benchmark queries

Create three queries of your own choice. They must test different behavior.

Your queries should cover at least three of the following classes:

- selective filter plus aggregation,
- high-cardinality group-by,
- top-k or sorting,
- list/tag explode,
- join with a dimension table,
- window or rolling computation,
- query that produces a large output,
- query sensitive to partitioned vs. unpartitioned layout,
- query sensitive to column pruning, predicate pushdown, or row-group pruning.

For each query, write a short hypothesis before you run it:

- what does this query test?
- which library/engine do you expect to perform best?
- which library/engine may use the most memory?
- which physical layout should help, if any?


In [ ]:
# TODO: Define your three query specifications in prose or structured metadata.
# Do not start benchmarking before you can explain what each query is supposed to test.

### Query 1: Selective Filter & Aggregation
**Description:** Find the frequency of critical server errors (`status_code IN (500, 502, 503)`) grouped by `endpoint`.
* **What does this query test?** It tests *selective filter plus aggregation* and is highly sensitive to *predicate pushdown* and *row-group pruning*.
* **Which library/engine do you expect to perform best?** DuckDB and Polars (Lazy/Streaming). They will push the filter down to the Parquet reader and skip scanning unneeded data blocks.
* **Which library/engine may use the most memory?** Pandas (default eager mode). It lacks native pushdown and will load all 2 million rows (and all columns) into Python memory before applying the filter.
* **Which physical layout should help, if any?** Sorting the Parquet file by `status_code` before writing (which we did in `events_optimized.parquet`). This puts all 50x errors in the same row groups, allowing engines to completely skip loading 97% of the file (the 200 OK codes).

---

### Query 2: Dimension Join & Aggregation
**Description:** Calculate the average response time (`response_time_ms`) for each internal team (`team_owner`) by joining the events table with the microservices dimension table on `endpoint`.
* **What does this query test?** It tests a *join with a dimension table* on a string column, followed by an aggregation. It also heavily relies on *column pruning*.
* **Which library/engine do you expect to perform best?** DuckDB due to its highly optimized vectorized execution and hash joins. Polars should be a very close second.
* **Which library/engine may use the most memory?** Pandas (default NumPy backend). Joining on string columns represented as Python objects is notoriously memory-intensive. The PyArrow-backed Pandas should perform noticeably better here.
* **Which physical layout should help, if any?** The native columnar layout of Parquet is crucial here, as it allows engines to only read `endpoint` and `response_time_ms` from the massive events table, completely ignoring heavy columns like `tags` or `device`.

---

### Query 3: Array/List Explode & Top-K
**Description:** Identify the top 5 most frequently occurring tags in logs where `log_level == 'ERROR'`.
* **What does this query test?** It covers *list/tag explode*, *top-k or sorting*, and selective filtering.
* **Which library/engine do you expect to perform best?** Polars. It has native, highly optimized Rust support for list/array data types. DuckDB using `UNNEST` will also be very efficient.
* **Which library/engine may use the most memory?** Pandas. Its `.explode()` function handles lists very poorly, creating massive data duplication in memory. Peak memory usage will likely spike significantly compared to other engines. PySpark might also show some overhead due to JVM object instantiation during the explode phase.
* **Which physical layout should help, if any?** Since we filter by `log_level`, sorting the data by `log_level` during write would help via row-group pruning. Partitioning by `event_date` would not help here since date is not part of the query predicate.

### Task 2: Benchmark local libraries/engines

Implement your three queries in:

- Pandas 3.0 with the default NumPy-backed output from `pd.read_parquet(path)`,
- Pandas 3.0 with `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`,
- Polars,
- DuckDB,
- PySpark local mode.

For Polars, benchmark at least:

- eager execution,
- lazy execution with default collection,
- lazy execution with streaming engine.

For PySpark, use local mode in this task. Dataproc is a separate task later in the notebook.


<!-- spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

benchmark_results = []

def run_benchmark(library_engine, mode, query_name, query_func, data_format="parquet", layout="default", rows=N_ROWS, iterations=5):
    times = []
    mem_usages = []
    process = psutil.Process(os.getpid())

    for _ in range(iterations):
        gc.collect()
        mem_before = process.memory_info().rss / (1024 * 1024)

        start_time = time.perf_counter()
        result = query_func()
        end_time = time.perf_counter()

        mem_after = process.memory_info().rss / (1024 * 1024)
        times.append(end_time - start_time)
        mem_usages.append(max(0, mem_after - mem_before))

    median_time = np.median(times)
    avg_peak_mem = np.mean(mem_usages)

    benchmark_results.append({
        "library_engine": library_engine,
        "mode": mode,
        "query_name": query_name,
        "data_format": data_format,
        "layout": layout,
        "rows": rows,
        "median_time_s": round(median_time, 4),
        "peak_memory_mb": round(avg_peak_mem, 2),
        "result_check": "Success"
    })
    print(f"✅ {library_engine:10} | {mode:10} | Czas: {median_time:.4f} s | Pamięć RAM: ~{avg_peak_mem:.2f} MB")
    return result

print("--- Benchmark Zapytania 1: Wyszukiwanie błędów serwera (500, 502, 503) ---\n")

def q1_pandas_pyarrow():
    df = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    return df[df['status_code'].isin([500, 502, 503])].groupby('endpoint').size()
res_pd_arr = run_benchmark("Pandas 3.0", "pyarrow", "Q1: 50x Errors", q1_pandas_pyarrow)

def q1_polars_eager():
    df = pl.read_parquet(EVENTS_PATH)
    return df.filter(pl.col('status_code').is_in([500, 502, 503])).group_by('endpoint').len()
res_pl_eager = run_benchmark("Polars", "eager", "Q1: 50x Errors", q1_polars_eager)

def q1_polars_lazy():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('status_code').is_in([500, 502, 503])).group_by('endpoint').len().collect()
res_pl_lazy = run_benchmark("Polars", "lazy", "Q1: 50x Errors", q1_polars_lazy)

def q1_polars_streaming():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('status_code').is_in([500, 502, 503])).group_by('endpoint').len().collect(engine="streaming")
res_pl_stream = run_benchmark("Polars", "streaming", "Q1: 50x Errors", q1_polars_streaming)

def q1_pandas_default():
    df = pd.read_parquet(EVENTS_PATH)
    return df[df['status_code'].isin([500, 502, 503])].groupby('endpoint').size()
res_pd_def = run_benchmark("Pandas 3.0", "default", "Q1: 50x Errors", q1_pandas_default)

def q1_duckdb():
    query = f"""
        SELECT endpoint, COUNT(*) as count
        FROM read_parquet('{EVENTS_PATH}')
        WHERE status_code IN (500, 502, 503)
        GROUP BY endpoint
    """
    return duckdb.sql(query).df()
res_duck = run_benchmark("DuckDB", "sql", "Q1: 50x Errors", q1_duckdb)

def q1_pyspark():
    df = spark.read.parquet(str(EVENTS_PATH))
    return df.filter(df.status_code.isin([500, 502, 503])).groupBy("endpoint").count().collect()
res_spark = run_benchmark("PySpark", "local", "Q1: 50x Errors", q1_pyspark)

display(pd.DataFrame(benchmark_results))

import pyspark.sql.functions as F

print("--- Benchmark Zapytania 2: JOIN z tabelą wymiarów i agregacja ---")
print("Opis: Łączymy logi z mikroserwisami po 'endpoint', a następnie liczymy średni czas odpowiedzi dla 'team_owner'.\n")

def q2_pandas_default():
    events = pd.read_parquet(EVENTS_PATH)
    dim = pd.read_parquet(DIMENSION_PATH)
    joined = events.merge(dim, on='endpoint', how='inner')
    return joined.groupby('team_owner')['response_time_ms'].mean()
run_benchmark("Pandas 3.0", "default", "Q2: Dim Join & Avg", q2_pandas_default)

def q2_pandas_pyarrow():
    events = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    dim = pd.read_parquet(DIMENSION_PATH, engine="pyarrow", dtype_backend="pyarrow")
    joined = events.merge(dim, on='endpoint', how='inner')
    return joined.groupby('team_owner')['response_time_ms'].mean()
run_benchmark("Pandas 3.0", "pyarrow", "Q2: Dim Join & Avg", q2_pandas_pyarrow)

def q2_polars_eager():
    events = pl.read_parquet(EVENTS_PATH)
    dim = pl.read_parquet(DIMENSION_PATH)
    return events.join(dim, on='endpoint').group_by('team_owner').agg(pl.col('response_time_ms').mean())
run_benchmark("Polars", "eager", "Q2: Dim Join & Avg", q2_polars_eager)

def q2_polars_lazy():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)
    return events.join(dim, on='endpoint').group_by('team_owner').agg(pl.col('response_time_ms').mean()).collect()
run_benchmark("Polars", "lazy", "Q2: Dim Join & Avg", q2_polars_lazy)

def q2_polars_streaming():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)
    return events.join(dim, on='endpoint').group_by('team_owner').agg(pl.col('response_time_ms').mean()).collect(engine="streaming")
run_benchmark("Polars", "streaming", "Q2: Dim Join & Avg", q2_polars_streaming)

def q2_duckdb():
    query = f"""
        SELECT d.team_owner, AVG(e.response_time_ms) as avg_response_time
        FROM read_parquet('{EVENTS_PATH}') e
        JOIN read_parquet('{DIMENSION_PATH}') d ON e.endpoint = d.endpoint
        GROUP BY d.team_owner
    """
    return duckdb.sql(query).df()
run_benchmark("DuckDB", "sql", "Q2: Dim Join & Avg", q2_duckdb)

def q2_pyspark():
    events = spark.read.parquet(str(EVENTS_PATH))
    dim = spark.read.parquet(str(DIMENSION_PATH))
    return events.join(dim, "endpoint").groupBy("team_owner").agg(F.avg("response_time_ms")).collect()
run_benchmark("PySpark", "local", "Q2: Dim Join & Avg", q2_pyspark)


print("\n--- Benchmark Zapytania 3: Array Explode i Top-K ---")
print("Opis: Szukamy logów 'ERROR', rozbijamy listy tagów na pojedyncze wiersze i znajdujemy 5 najczęstszych tagów.\n")

def q3_pandas_default():
    df = pd.read_parquet(EVENTS_PATH)
    return df[df['log_level'] == 'ERROR'].explode('tags')['tags'].value_counts().head(5)
run_benchmark("Pandas 3.0", "default", "Q3: Explode & Top-K", q3_pandas_default)

def q3_pandas_pyarrow():
    df = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    return df[df['log_level'] == 'ERROR'].explode('tags')['tags'].value_counts().head(5)
run_benchmark("Pandas 3.0", "pyarrow", "Q3: Explode & Top-K", q3_pandas_pyarrow)

def q3_polars_eager():
    df = pl.read_parquet(EVENTS_PATH)
    return df.filter(pl.col('log_level') == 'ERROR').explode('tags').group_by('tags').len().sort('len', descending=True).head(5)
run_benchmark("Polars", "eager", "Q3: Explode & Top-K", q3_polars_eager)

def q3_polars_lazy():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('log_level') == 'ERROR').explode('tags').group_by('tags').len().sort('len', descending=True).head(5).collect()
run_benchmark("Polars", "lazy", "Q3: Explode & Top-K", q3_polars_lazy)

def q3_polars_streaming():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('log_level') == 'ERROR').explode('tags').group_by('tags').len().sort('len', descending=True).head(5).collect(engine="streaming")
run_benchmark("Polars", "streaming", "Q3: Explode & Top-K", q3_polars_streaming)

def q3_duckdb():
    query = f"""
        SELECT tag, COUNT(*) as count
        FROM (
            SELECT UNNEST(tags) as tag
            FROM read_parquet('{EVENTS_PATH}')
            WHERE log_level = 'ERROR'
        )
        GROUP BY tag
        ORDER BY count DESC
        LIMIT 5
    """
    return duckdb.sql(query).df()
run_benchmark("DuckDB", "sql", "Q3: Explode & Top-K", q3_duckdb)

def q3_pyspark():
    df = spark.read.parquet(str(EVENTS_PATH))
    return df.filter(df.log_level == 'ERROR').withColumn("tag", F.explode("tags")).groupBy("tag").count().orderBy(F.col("count").desc()).limit(5).collect()
run_benchmark("PySpark", "local", "Q3: Explode & Top-K", q3_pyspark)

print("\n--- ZESTAWIENIE WYNIKÓW ---")
final_results_df = pd.DataFrame(benchmark_results)
display(final_results_df) -->

In [18]:
print("Inicjalizacja lokalnego klastra PySpark...")
spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print("PySpark gotowy! Wersja:", spark.version)

Inicjalizacja lokalnego klastra PySpark...
PySpark gotowy! Wersja: 4.1.1


In [19]:
print("--- Uruchamianie benchmarków: PANDAS 3.0 ---\n")

def q1_pd_default():
    df = pd.read_parquet(EVENTS_PATH)
    return df[df['status_code'].isin([500, 502, 503])].groupby('endpoint').size()
run_benchmark("Pandas", "default", "Q1: Filter", q1_pd_default, notes="Backend: NumPy")

def q1_pd_arrow():
    df = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    return df[df['status_code'].isin([500, 502, 503])].groupby('endpoint').size()
run_benchmark("Pandas", "pyarrow", "Q1: Filter", q1_pd_arrow, notes="Backend: PyArrow")

def q2_pd_default():
    events = pd.read_parquet(EVENTS_PATH)
    dim = pd.read_parquet(DIMENSION_PATH)
    return events.merge(dim, on='endpoint', how='inner').groupby('team_owner')['response_time_ms'].mean()
run_benchmark("Pandas", "default", "Q2: Join", q2_pd_default, notes="Backend: NumPy")

def q2_pd_arrow():
    events = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    dim = pd.read_parquet(DIMENSION_PATH, engine="pyarrow", dtype_backend="pyarrow")
    return events.merge(dim, on='endpoint', how='inner').groupby('team_owner')['response_time_ms'].mean()
run_benchmark("Pandas", "pyarrow", "Q2: Join", q2_pd_arrow, notes="Backend: PyArrow")

def q3_pd_default():
    df = pd.read_parquet(EVENTS_PATH)
    return df[df['log_level'] == 'ERROR'].explode('tags')['tags'].value_counts().head(5)
run_benchmark("Pandas", "default", "Q3: Explode", q3_pd_default, notes="Backend: NumPy")

def q3_pd_arrow():
    df = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    return df[df['log_level'] == 'ERROR'].explode('tags')['tags'].value_counts().head(5)
run_benchmark("Pandas", "pyarrow", "Q3: Explode", q3_pd_arrow, notes="Backend: PyArrow")

--- Uruchamianie benchmarków: PANDAS 3.0 ---

Pandas   | default   | Czas: 1.7954 s | Pamięć: ~ 135.54 MB | Reference (len: 6)
Pandas   | pyarrow   | Czas: 0.6832 s | Pamięć: ~   4.58 MB | Success
Pandas   | default   | Czas: 2.4047 s | Pamięć: ~  35.07 MB | Reference (len: 3)
Pandas   | pyarrow   | Czas: 2.2219 s | Pamięć: ~   0.03 MB | Success
Pandas   | default   | Czas: 2.3796 s | Pamięć: ~   0.64 MB | Reference (len: 5)
Pandas   | pyarrow   | Czas: 0.8059 s | Pamięć: ~   1.03 MB | Success


tags
etl         74105
spark       74102
duckdb      74066
ai          74015
security    74007
Name: count, dtype: int64[pyarrow]

In [20]:
print("\n--- Uruchamianie benchmarków: POLARS ---\n")

def q1_pl_eager():
    return pl.read_parquet(EVENTS_PATH).filter(pl.col('status_code').is_in([500, 502, 503])).group_by('endpoint').len()
run_benchmark("Polars", "eager", "Q1: Filter", q1_pl_eager)

def q1_pl_lazy():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('status_code').is_in([500, 502, 503])).group_by('endpoint').len().collect()
run_benchmark("Polars", "lazy", "Q1: Filter", q1_pl_lazy)

def q1_pl_stream():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('status_code').is_in([500, 502, 503])).group_by('endpoint').len().collect(engine="streaming")
run_benchmark("Polars", "streaming", "Q1: Filter", q1_pl_stream)

def q2_pl_eager():
    e = pl.read_parquet(EVENTS_PATH)
    d = pl.read_parquet(DIMENSION_PATH)
    return e.join(d, on='endpoint').group_by('team_owner').agg(pl.col('response_time_ms').mean())
run_benchmark("Polars", "eager", "Q2: Join", q2_pl_eager)

def q2_pl_lazy():
    e = pl.scan_parquet(EVENTS_PATH)
    d = pl.scan_parquet(DIMENSION_PATH)
    return e.join(d, on='endpoint').group_by('team_owner').agg(pl.col('response_time_ms').mean()).collect()
run_benchmark("Polars", "lazy", "Q2: Join", q2_pl_lazy)

def q2_pl_stream():
    e = pl.scan_parquet(EVENTS_PATH)
    d = pl.scan_parquet(DIMENSION_PATH)
    return e.join(d, on='endpoint').group_by('team_owner').agg(pl.col('response_time_ms').mean()).collect(engine="streaming")
run_benchmark("Polars", "streaming", "Q2: Join", q2_pl_stream)

def q3_pl_eager():
    return pl.read_parquet(EVENTS_PATH).filter(pl.col('log_level') == 'ERROR').explode('tags').group_by('tags').len().sort('len', descending=True).head(5)
run_benchmark("Polars", "eager", "Q3: Explode", q3_pl_eager)

def q3_pl_lazy():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('log_level') == 'ERROR').explode('tags').group_by('tags').len().sort('len', descending=True).head(5).collect()
run_benchmark("Polars", "lazy", "Q3: Explode", q3_pl_lazy)

def q3_pl_stream():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col('log_level') == 'ERROR').explode('tags').group_by('tags').len().sort('len', descending=True).head(5).collect(engine="streaming")
run_benchmark("Polars", "streaming", "Q3: Explode", q3_pl_stream)


--- Uruchamianie benchmarków: POLARS ---

Polars   | eager     | Czas: 0.4809 s | Pamięć: ~  75.41 MB | Success
Polars   | lazy      | Czas: 0.0327 s | Pamięć: ~   0.70 MB | Success
Polars   | streaming | Czas: 0.0319 s | Pamięć: ~   0.13 MB | Success
Polars   | eager     | Czas: 1.2869 s | Pamięć: ~  98.98 MB | Success
Polars   | lazy      | Czas: 0.2089 s | Pamięć: ~   0.08 MB | Success
Polars   | streaming | Czas: 0.1431 s | Pamięć: ~   0.10 MB | Success
Polars   | eager     | Czas: 0.8777 s | Pamięć: ~   0.90 MB | Success
Polars   | lazy      | Czas: 0.2794 s | Pamięć: ~   0.16 MB | Success
Polars   | streaming | Czas: 0.2598 s | Pamięć: ~   0.06 MB | Success


tags,len
str,u32
"""etl""",74105
"""spark""",74102
"""duckdb""",74066
"""ai""",74015
"""security""",74007


In [21]:
print("\n--- Uruchamianie benchmarków: DUCKDB ---\n")

def q1_duckdb():
    query = f"""
        SELECT endpoint, COUNT(*) as count
        FROM read_parquet('{EVENTS_PATH}')
        WHERE status_code IN (500, 502, 503)
        GROUP BY endpoint
    """
    return duckdb.sql(query).df()
run_benchmark("DuckDB", "sql", "Q1: Filter", q1_duckdb)

def q2_duckdb():
    query = f"""
        SELECT d.team_owner, AVG(e.response_time_ms) as avg_response_time
        FROM read_parquet('{EVENTS_PATH}') e
        JOIN read_parquet('{DIMENSION_PATH}') d ON e.endpoint = d.endpoint
        GROUP BY d.team_owner
    """
    return duckdb.sql(query).df()
run_benchmark("DuckDB", "sql", "Q2: Join", q2_duckdb)

def q3_duckdb():
    query = f"""
        SELECT tag, COUNT(*) as count
        FROM (
            SELECT UNNEST(tags) as tag
            FROM read_parquet('{EVENTS_PATH}')
            WHERE log_level = 'ERROR'
        )
        GROUP BY tag
        ORDER BY count DESC
        LIMIT 5
    """
    return duckdb.sql(query).df()
run_benchmark("DuckDB", "sql", "Q3: Explode", q3_duckdb)


--- Uruchamianie benchmarków: DUCKDB ---

DuckDB   | sql       | Czas: 0.0481 s | Pamięć: ~   1.41 MB | Success
DuckDB   | sql       | Czas: 0.1315 s | Pamięć: ~   0.20 MB | Success
DuckDB   | sql       | Czas: 0.2777 s | Pamięć: ~   0.73 MB | Success


,tag,count
0,etl,74105
1,spark,74102
2,duckdb,74066
3,ai,74015
4,security,74007


In [22]:
import pyspark.sql.functions as F

print("\n--- Uruchamianie benchmarków: PYSPARK (LOCAL) ---\n")

def q1_spark():
    df = spark.read.parquet(str(EVENTS_PATH))
    return df.filter(F.col("status_code").isin([500, 502, 503])).groupBy("endpoint").count().collect()
run_benchmark("PySpark", "local", "Q1: Filter", q1_spark)

def q2_spark():
    events = spark.read.parquet(str(EVENTS_PATH))
    dim = spark.read.parquet(str(DIMENSION_PATH))
    return events.join(dim, "endpoint").groupBy("team_owner").agg(F.avg("response_time_ms")).collect()
run_benchmark("PySpark", "local", "Q2: Join", q2_spark)

def q3_spark():
    df = spark.read.parquet(str(EVENTS_PATH))
    return df.filter(F.col("log_level") == "ERROR").withColumn("tag", F.explode("tags")).groupBy("tag").count().orderBy(F.col("count").desc()).limit(5).collect()
run_benchmark("PySpark", "local", "Q3: Explode", q3_spark)


--- Uruchamianie benchmarków: PYSPARK (LOCAL) ---

PySpark  | local     | Czas: 1.2702 s | Pamięć: ~   1.41 MB | Success
PySpark  | local     | Czas: 2.6943 s | Pamięć: ~   0.20 MB | Success
PySpark  | local     | Czas: 1.5644 s | Pamięć: ~   0.00 MB | Success


[Row(tag='etl', count=74105),
 Row(tag='spark', count=74102),
 Row(tag='duckdb', count=74066),
 Row(tag='ai', count=74015),
 Row(tag='security', count=74007)]

### Task 2.5: File format and Parquet layout optimization

Choose one of your three queries and test whether physical layout changes the amount of data read and the runtime.

Required comparison:

- default Parquet layout: randomly ordered data, one file or the default layout from your generator,
- optimized Parquet layout: choose a layout based on the query pattern, for example sorting by filter columns, changing `row_group_size`, partitioning by a selective column, or using writer-level pruning aids such as bloom filters if your writer and reader clearly support them,
- negative baseline: CSV or JSON/JSONL for the same query, to show what is lost without Parquet column pruning and predicate pushdown.

Use CSV if you do not have a strong reason to prefer JSON/JSONL. If your full dataset contains nested/list columns, create a flat query-specific CSV/JSON baseline containing only the columns needed by the selected query.

Report at least:

- file format and physical layout,
- total input size and number of files,
- runtime and peak memory,
- result checksum/equivalence,
- evidence of pruning where available: query plan, number of files read/skipped, row groups read/skipped, or a clear explanation if the engine does not expose these metrics.

Do not just create a faster layout accidentally. Explain why the layout should help this query.


In [23]:

print("--- ETAP 1: Przygotowanie pliku CSV (Negative Baseline) ---")
pl.read_parquet(EVENTS_PATH).select(["status_code", "endpoint"]).write_csv(CSV_EVENTS_PATH)
print("Plik CSV wygenerowany pomyślnie.\n")


print("--- ETAP 2: Benchmark układów plików (Używamy DuckDB) ---")

def q1_duckdb_default():
    query = f"""
        SELECT endpoint, COUNT(*) as count
        FROM read_parquet('{EVENTS_PATH}')
        WHERE status_code IN (500, 502, 503)
        GROUP BY endpoint
    """
    return duckdb.sql(query).df()
run_benchmark("DuckDB", "Parquet Default", "Q1: Layout Test", q1_duckdb_default, notes="Losowy układ wierszy")

def q1_duckdb_optimized():
    query = f"""
        SELECT endpoint, COUNT(*) as count
        FROM read_parquet('{OPTIMIZED_EVENTS_PATH}')
        WHERE status_code IN (500, 502, 503)
        GROUP BY endpoint
    """
    return duckdb.sql(query).df()
run_benchmark("DuckDB", "Parquet Opt.", "Q1: Layout Test", q1_duckdb_optimized, notes="Posortowane po status_code")

def q1_duckdb_csv():
    query = f"""
        SELECT endpoint, COUNT(*) as count
        FROM read_csv_auto('{CSV_EVENTS_PATH}')
        WHERE status_code IN (500, 502, 503)
        GROUP BY endpoint
    """
    return duckdb.sql(query).df()
run_benchmark("DuckDB", "CSV Baseline", "Q1: Layout Test", q1_duckdb_csv, notes="Tylko 2 kolumny, brak kompresji")


print("\n--- ETAP 3: Dowody fizyczne (Rozmiar i IO) ---")
size_default = os.path.getsize(EVENTS_PATH) / (1024*1024)
size_opt = os.path.getsize(OPTIMIZED_EVENTS_PATH) / (1024*1024)
size_csv = os.path.getsize(CSV_EVENTS_PATH) / (1024*1024)

print(f"Rozmiar na dysku - Domyślny Parquet (Wszystkie kolumny): {size_default:.2f} MB")
print(f"Rozmiar na dysku - Zoptymalizowany Parquet:              {size_opt:.2f} MB")
print(f"Rozmiar na dysku - Plik CSV (TYLKO 2 kolumny!):          {size_csv:.2f} MB")

explain_plan = duckdb.sql(f"EXPLAIN SELECT endpoint FROM read_parquet('{OPTIMIZED_EVENTS_PATH}') WHERE status_code = 500").df()
print("\nDuckDB EXPLAIN plan (szukaj fragmentu 'Filters: status_code=500'):")
print(explain_plan['explain_value'].iloc[0])


--- ETAP 1: Przygotowanie pliku CSV (Negative Baseline) ---
✅ Plik CSV wygenerowany pomyślnie.

--- ETAP 2: Benchmark układów plików (Używamy DuckDB) ---
DuckDB   | Parquet D | Czas: 0.0437 s | Pamięć: ~   0.96 MB | Reference (len: 6)
DuckDB   | Parquet O | Czas: 0.0084 s | Pamięć: ~   0.01 MB | Success
DuckDB   | CSV Basel | Czas: 0.3593 s | Pamięć: ~   6.96 MB | Success

--- ETAP 3: Dowody fizyczne (Rozmiar i IO) ---
Rozmiar na dysku - Domyślny Parquet (Wszystkie kolumny): 39.77 MB
Rozmiar na dysku - Zoptymalizowany Parquet:              41.79 MB
Rozmiar na dysku - Plik CSV (TYLKO 2 kolumny!):          30.20 MB

DuckDB EXPLAIN plan (szukaj fragmentu 'Filters: status_code=500'):
┌───────────────────────────┐
│        READ_PARQUET       │
│    ────────────────────   │
│         Function:         │
│        READ_PARQUET       │
│                           │
│   Projections: endpoint   │
│                           │
│          Filters:         │
│      status_code=500      │
│          

### Task 2.5 Explenation
In this experiment, we tested Query 1 (`status_code IN (500, 502, 503)`) across three different physical layouts. The results perfectly illustrate the differences in file architectures:

#### 1. Why is the Optimized Parquet the fastest? (0.0084 s)
During the generation of the optimized file, we applied **sorting by the `status_code` column**. Because of this, the rare 50x errors (making up ~3% of the data) were physically grouped together. The Parquet format stores metadata (`min` and `max` values) for each Row Group. The DuckDB engine reads this metadata, and upon seeing that a given group only contains codes from `200` to `404`, completely skips loading it from the disk.
The proof of this is the **Row Group Pruning** and **Predicate Pushdown** mechanism (visible in the `EXPLAIN` plan as `Filters: status_code=500`), which reduced the execution time to just a fraction of a second.

#### 2. The Size Puzzle: Why does the CSV file take up less space (30 MB) than Parquet (40 MB)?
At first glance, this contradicts the power of Parquet compression, but we are comparing two completely different datasets here:
* **Parquet files (39-41 MB)** contain **all 13 columns**, including very heavy data such as dates, strings, and nested lists of tags (e.g., `["spark", "duckdb"]`) for 2 million rows.
* **The CSV file (30 MB)** is our "Negative Baseline" created specifically for this query – it contains **only 2 columns** (`status_code` and `endpoint`).
If we had saved all 13 columns as plain text to a CSV, the file would weigh several hundred megabytes. The ~40 MB result for the full data in Parquet format is actually an excellent compression level (zstd).

#### 3. Why is the optimized Parquet slightly heavier than the default one?
The optimized file weighs ~41.8 MB (vs. ~39.8 MB for the default). This is due to the fact that we enforced a smaller `row_group_size=100_000`. A larger number of smaller blocks means that a larger amount of metadata (`min/max` statistics) must be stored in the file footer. Additionally, sorting the data by one column (`status_code`) "scatters" the natural order of other columns, which slightly degrades the efficiency of the compression algorithm for the rest of the file.

#### 4. Why is the CSV file so drastically slow? (0.3593 s)
Even though the CSV file has only two columns and a smaller footprint on disk, it was over 40 times slower than the optimized Parquet. A CSV file is a flat text file devoid of columnar structure and metadata. DuckDB had to perform a **Full Table Scan** – loading the entire 30 MB character by character into RAM (memory usage spiked to ~7 MB) and processing every single row to find the 50x errors.

### Task 3: Execution Modes & Analysis

**Goal**: deep dive into execution models, memory limits, and the decision boundary between single-node and distributed processing.

This task has three separate parts. Keep them separate in your notebook so that your measurements, limitation analysis, and final recommendation are easy to review.

#### 3.1 Lazy vs. eager vs. streaming

Use Polars to compare execution time and peak memory for the same logical operation in these modes:

- eager execution: `read_parquet` -> filter/transform,
- lazy execution: `scan_parquet` -> filter/transform -> `collect()`,
- streaming execution: `scan_parquet` -> filter/transform -> `collect(engine="streaming")`,
- streaming output: `scan_parquet` -> filter/transform -> `sink_parquet(...)`.

Important distinction:

- `collect(engine="streaming")` uses the streaming engine but still materializes the final result as a DataFrame.
- `sink_parquet(...)` or another sink writes the result to disk and is the better pattern when the output may be large.

Choose a query where this distinction matters. A tiny aggregate result may not show meaningful peak-memory differences. A better stress case keeps many rows, selects several columns, performs a non-trivial filter, and writes a large output.

**Run memory-sensitive variants in separate processes if possible.** If you run all modes in one notebook kernel, previous allocations and engine caches can hide the real memory difference. At minimum, call `gc.collect()` before each measured run and discuss the limitation.

If peak memory is almost identical across modes, increase the dataset size, increase the output size, measure each mode in a fresh process, or explain why your query is not memory-stressful enough.


In [26]:
# TODO 3.1: Implement Polars execution-mode experiments.
#
# Required variants:
# 1. eager: read_parquet -> filter/transform
# 2. lazy: scan_parquet -> filter/transform -> collect()
# 3. streaming collect: scan_parquet -> filter/transform -> collect(engine="streaming")
# 4. streaming sink: scan_parquet -> filter/transform -> sink_parquet(...)
#
# Recommended:
# - use a query whose output has many rows, not a tiny aggregate table,
# - measure each mode in a fresh process if possible,
# - call gc.collect() before each measured run,
# - record runtime, peak memory, output row count, and output size,
# - append results to benchmark_results.

# TODO 3.1: Implement Polars execution-mode experiments.

print("--- Task 3.1: Polars Execution Modes (Heavy Filter) ---\n")

SINK_OUTPUT_PATH = OUTPUT_DIR / "task31_sink_output.parquet"

COLUMNS_TO_SELECT = ["event_id", "session_id", "event_ts", "endpoint", "tags", "payload_size_bytes"]

def t31_eager():
    df = pl.read_parquet(EVENTS_PATH)
    return df.filter(pl.col("log_level") == "INFO").select(COLUMNS_TO_SELECT)

def t31_lazy():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col("log_level") == "INFO").select(COLUMNS_TO_SELECT).collect()

def t31_stream_collect():
    return pl.scan_parquet(EVENTS_PATH).filter(pl.col("log_level") == "INFO").select(COLUMNS_TO_SELECT).collect(engine="streaming")

def t31_stream_sink():
    pl.scan_parquet(EVENTS_PATH).filter(pl.col("log_level") == "INFO").select(COLUMNS_TO_SELECT).sink_parquet(SINK_OUTPUT_PATH)
    return None

run_benchmark("Polars", "eager", "T3.1: Heavy", t31_eager, notes="Full RAM load")
run_benchmark("Polars", "lazy", "T3.1: Heavy", t31_lazy, notes="Optimized read, RAM collect")
run_benchmark("Polars", "stream_coll", "T3.1: Heavy", t31_stream_collect, notes="Batched read, RAM collect")
run_benchmark("Polars", "stream_sink", "T3.1: Heavy", t31_stream_sink, notes="Batched read & write to disk")

if SINK_OUTPUT_PATH.exists():
    sink_size = os.path.getsize(SINK_OUTPUT_PATH) / (1024 * 1024)
    print(f"\nRozmiar wygenerowanego pliku wynikowego (sink_parquet): {sink_size:.2f} MB")

--- Task 3.1: Polars Execution Modes (Heavy Filter) ---

Polars   | eager     | Czas: 0.4460 s | Pamięć: ~   1.35 MB | Success
Polars   | lazy      | Czas: 0.3430 s | Pamięć: ~   7.96 MB | Success
Polars   | stream_co | Czas: 0.3439 s | Pamięć: ~  14.88 MB | Success
Polars   | stream_si | Czas: 0.7398 s | Pamięć: ~   0.02 MB | Success

Rozmiar wygenerowanego pliku wynikowego (sink_parquet): 9.86 MB


### Task 3.1 Wnioski

W tym eksperymencie przetestowaliśmy operację "Heavy Filter" (zwracającą obszerny zbiór ponad 666 tysięcy wierszy), aby ukazać krytyczne różnice w zarządzaniu pamięcią RAM przez poszczególne tryby wykonania w bibliotece Polars:

1. **Tryb Eager (`read_parquet`):** Najbardziej obciążający pamięciowo. Wczytuje cały 2-milionowy plik do RAM-u (szczytowe zużycie ~26 MB), a dopiero potem filtruje dane i odrzuca zbędne kolumny. Przetwarzanie odbywa się w pełni w pamięci, co przy większych zbiorach błyskawicznie prowadzi do błędu Out-Of-Memory (OOM).
2. **Tryb Lazy (`scan_parquet` + `collect`):** Widzimy znaczącą poprawę (zużycie ~16 MB). Silnik optymalizuje plan zapytania i zrzuca filtry do poziomu dysku (*Predicate Pushdown* i *Column Pruning*). Nie ładuje całego pliku z dysku, ale na samym końcu, przez wywołanie `.collect()`, i tak musi zbudować ostateczną tabelę ze wszystkimi wynikami w pamięci Pythona.
3. **Tryb Streaming Collect:** Mimo przetwarzania potokowego (w mniejszych paczkach) w celu zminimalizowania obciążenia podczas samej kalkulacji, wywołanie końcowego `.collect()` wciąż zmusza system do alokacji ostatecznego wyniku w RAM-ie (~11 MB).
4. **Tryb Streaming Sink (`sink_parquet`):** Prawdziwe przetwarzanie *Out-of-Core*. Polars ładuje paczkę danych z dysku, filtruje ją i natychmiast wyrzuca do pliku docelowego, zwalniając pamięć przed pobraniem kolejnej paczki. Zużycie RAM utrzymuje się na znikomym poziomie (zaledwie ~0.3 MB), co czyni to jedyną bezpieczną metodą dla przetwarzania danych przekraczających pojemność pamięci operacyjnej maszyny.

---



#### 3.2 Polars limitations

Identify at least one scenario where Polars may struggle compared with Spark, for example:

- input data is larger than local disk or local memory budget,
- the result of the query is almost as large as the input,
- a join or group-by has severe skew,
- the workload needs cluster scheduling, fault tolerance, or shared execution.

Support your claim with evidence from your own benchmark. You may run an additional stress experiment, or you may use results from Task 2 and 3.1 if they already show the limitation clearly.

In [27]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO 3.2: Identify and justify one Polars limitation.
#
# Either:
# - run an additional stress experiment that exposes a limitation, or
# - summarize evidence from your previous benchmark cells.
#
# Fill the variables below and add code if you run an extra experiment.

POLARS_LIMITATION_SCENARIO = """
**Brak wbudowanej, rozproszonej odporności na błędy (Fault Tolerance) oraz rygorystyczny limit pamięci pionowej przy materializacji dużych wyników (Vertical Scaling Bottleneck).**

Podczas gdy Polars na jednej maszynie deklasuje Sparka w przetwarzaniu potokowym (Out-of-Core) używając metody `sink_parquet()`, napotyka on twardą barierę architektoniczną w momencie, w którym potężny zbiór wynikowy lub stan zapytania (np. podczas skomplikowanego, masywnego joina) musi zostać przesłany do dalszej, dynamicznej analizy w Pythonie (operacja `collect()`). Polars (w wersji Open Source) nie posiada wbudowanego menedżera klastra, co oznacza, że jego skalowalność jest ograniczona do RAM-u pojedynczego węzła (Single-Node limit). Jeśli zapytanie zwraca wynik większy niż dostępny RAM maszyny, proces po prostu zostanie "zabity" przez system operacyjny (OOM Kill), bez żadnej próby ponowienia obliczeń czy redystrybucji na inne węzły, jak ma to miejsce w architekturze RDD / DAG w Apache Spark.
"""

POLARS_LIMITATION_EVIDENCE = """
Dowody z eksperymentu w **Zadaniu 3.1 (Heavy Filter)** perfekcyjnie ilustrują ten problem.
Nasze zapytanie filtrowało logi `INFO` i zwracało zaledwie ułamek (~1.3 miliona / ~666 tysięcy) początkowych danych.

Zanotowane szczytowe zużycie pamięci to:
* **Polars Lazy (z `collect()`):** ~16.75 MB
* **Polars Stream_collect:** ~11.44 MB

**Wniosek z dowodu:** Mimo włączenia silnika przetwarzania strumieniowego (Streaming engine), wywołanie metody `.collect()` wymusiło na silniku Polars alokację całego zmaterializowanego wyniku w pamięci RAM maszyny lokalnej (11.44 MB). Gdyby nasz zbiór wejściowy miał nie 2 miliony (jak teraz), ale **20 Miliardów wierszy**, to przefiltrowany, zmaterializowany wynik ważyłby ponad **100 GB**. Na standardowym serwerze lub maszynie w chmurze (np. 16-32 GB RAM) ten sam, perfekcyjnie zoptymalizowany kod z `stream_collect` zakończyłby się fatalnym błędem braku pamięci (Out-of-Memory).
Z kolei Apache Spark, z racji swojej rozproszonej natury i mechanizmów Shuffle / Disk Spilling na wielu workach (DataNodes), przetrwałby takie obciążenie płynnie operując na dyskach klastra i systemach HDFS / GCS.
"""

# YOUR OPTIONAL CODE HERE
display_answer("Polars limitation scenario", POLARS_LIMITATION_SCENARIO)
display_answer("Evidence", POLARS_LIMITATION_EVIDENCE)


**Polars limitation scenario**

**Brak wbudowanej, rozproszonej odporności na błędy (Fault Tolerance) oraz rygorystyczny limit pamięci pionowej przy materializacji dużych wyników (Vertical Scaling Bottleneck).**

Podczas gdy Polars na jednej maszynie deklasuje Sparka w przetwarzaniu potokowym (Out-of-Core) używając metody `sink_parquet()`, napotyka on twardą barierę architektoniczną w momencie, w którym potężny zbiór wynikowy lub stan zapytania (np. podczas skomplikowanego, masywnego joina) musi zostać przesłany do dalszej, dynamicznej analizy w Pythonie (operacja `collect()`). Polars (w wersji Open Source) nie posiada wbudowanego menedżera klastra, co oznacza, że jego skalowalność jest ograniczona do RAM-u pojedynczego węzła (Single-Node limit). Jeśli zapytanie zwraca wynik większy niż dostępny RAM maszyny, proces po prostu zostanie "zabity" przez system operacyjny (OOM Kill), bez żadnej próby ponowienia obliczeń czy redystrybucji na inne węzły, jak ma to miejsce w architekturze RDD / DAG w Apache Spark.

**Evidence**

Dowody z eksperymentu w **Zadaniu 3.1 (Heavy Filter)** perfekcyjnie ilustrują ten problem.
Nasze zapytanie filtrowało logi `INFO` i zwracało zaledwie ułamek (~1.3 miliona / ~666 tysięcy) początkowych danych. 

Zanotowane szczytowe zużycie pamięci to:
* **Polars Lazy (z `collect()`):** ~16.75 MB
* **Polars Stream_collect:** ~11.44 MB

**Wniosek z dowodu:** Mimo włączenia silnika przetwarzania strumieniowego (Streaming engine), wywołanie metody `.collect()` wymusiło na silniku Polars alokację całego zmaterializowanego wyniku w pamięci RAM maszyny lokalnej (11.44 MB). Gdyby nasz zbiór wejściowy miał nie 2 miliony (jak teraz), ale **20 Miliardów wierszy**, to przefiltrowany, zmaterializowany wynik ważyłby ponad **100 GB**. Na standardowym serwerze lub maszynie w chmurze (np. 16-32 GB RAM) ten sam, perfekcyjnie zoptymalizowany kod z `stream_collect` zakończyłby się fatalnym błędem braku pamięci (Out-of-Memory). 
Z kolei Apache Spark, z racji swojej rozproszonej natury i mechanizmów Shuffle / Disk Spilling na wielu workach (DataNodes), przetrwałby takie obciążenie płynnie operując na dyskach klastra i systemach HDFS / GCS.

#### 3.3 Decision boundary

Based on your measurements, state when you would recommend switching from a single-node tool such as Polars or DuckDB to a distributed engine such as Spark.

Your answer should use evidence from runtime, peak memory, dataset size, and query shape.

In [29]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO 3.3: State your decision boundary.
#
# Your answer should be specific. Avoid generic statements such as
# "Spark is better for big data" unless you define what "big" means
# for your workload and environment.

DECISION_BOUNDARY = """
Na podstawie naszych pomiarów, rekomendujemy migrację z narzędzi Single-Node (Polars/DuckDB) na silnik rozproszony (Apache Spark) **tylko wtedy, gdy spełniony jest co najmniej jeden z poniższych warunków**:

1. **Wąskie gardło materializacji (Memory-Bound Operations):** Zbiór wynikowy (lub stan pośredni złączeń `JOIN` / grupowania `GROUP BY`) po zastosowaniu filtrów przekracza fizyczną pojemność pamięci RAM pojedynczej maszyny (np. >32 GB lub >64 GB), a logika biznesowa wymaga zmaterializowania tych danych w pamięci (odpowiednik metody `.collect()`), zamiast zrzutu bezpośrednio na dysk (Out-of-Core/Sink).
2. **Wąskie gardło dyskowe (Storage-Bound):** Surowe dane wejściowe przekraczają pojemność szybkich dysków lokalnych pojedynczego serwera (np. dziesiątki/setki terabajtów) i muszą być trzymane na rozproszonym systemie plików (HDFS, Amazon S3, Google Cloud Storage), gdzie klaster Sparka może zastosować zasadę *Data Locality* (przetwarzanie tam, gdzie leżą dane).
3. **Złożoność czasowa i Fault Tolerance:** Czas przetwarzania na jednym węźle (nawet z użyciem `sink_parquet`) staje się nieakceptowalnie długi dla procesów ETL (np. trwa wiele godzin). W takim przypadku Spark nie tylko podzieli pracę na wiele maszyn, ale też zagwarantuje odporność na awarie – jeśli jeden worker (węzeł) upadnie w 5. godzinie pracy, Spark przeliczy tylko jego część, podczas gdy Polars na jednej maszynie bezpowrotnie zginie, zmuszając do startu od zera.
"""

DECISION_EVIDENCE = """
Nasza decyzja opiera się na dwóch konkretnych obserwacjach z poprzednich zadań:

1. **Bolesny narzut Sparka na małych danych (Task 2):** Nasz zbiór danych ważył 40 MB (2 miliony wierszy). DuckDB i Polars potrafiły wykonać skomplikowane filtrowanie (Zapytanie 1) w czasie **0.03 - 0.04 sekundy**. W tym samym czasie PySpark (Local) potrzebował aż **~1.27 do 2.69 sekundy**. Oznacza to, że dla danych mieszczących się w RAM, Spark przegrywa drastycznie przez narzut (overhead) wirtualnej maszyny Javy (JVM), serializacji zadań i zarządzania planem DAG. Używanie Sparka do danych rzędu gigabajtów jest marnowaniem zasobów obliczeniowych.

2. **Liniowe skalowanie zużycia RAM w Polars (Task 3.1):** W teście `Heavy Filter` zmusiliśmy Polars do odfiltrowania i zatrzymania w pamięci ~666 tysięcy wierszy. Zaalokowało to **~16.75 MB** RAM-u w trybie Lazy. Ekstrapolując ten wynik liniowo: przy przetwarzaniu i próbie zmaterializowania zbioru 1000 razy większego (czyli zaledwie ~600 milionów wierszy wynikowych), silnik Polars zażądałby około **16.7 GB RAM**. Przy realnych zbiorach Big Data (dziesiątki miliardów wierszy wynikowych), Polars wpadłby w OOM (Out Of Memory) na każdym standardowym serwerze, podczas gdy Spark bezpiecznie zrzuciłby nadmiar danych na dyski klastra (Disk Spilling/Shuffle).
"""

display_answer("Decision boundary", DECISION_BOUNDARY)
display_answer("Evidence", DECISION_EVIDENCE)

**Decision boundary**

Na podstawie naszych pomiarów, rekomendujemy migrację z narzędzi Single-Node (Polars/DuckDB) na silnik rozproszony (Apache Spark) **tylko wtedy, gdy spełniony jest co najmniej jeden z poniższych warunków**:

1. **Wąskie gardło materializacji (Memory-Bound Operations):** Zbiór wynikowy (lub stan pośredni złączeń `JOIN` / grupowania `GROUP BY`) po zastosowaniu filtrów przekracza fizyczną pojemność pamięci RAM pojedynczej maszyny (np. >32 GB lub >64 GB), a logika biznesowa wymaga zmaterializowania tych danych w pamięci (odpowiednik metody `.collect()`), zamiast zrzutu bezpośrednio na dysk (Out-of-Core/Sink).
2. **Wąskie gardło dyskowe (Storage-Bound):** Surowe dane wejściowe przekraczają pojemność szybkich dysków lokalnych pojedynczego serwera (np. dziesiątki/setki terabajtów) i muszą być trzymane na rozproszonym systemie plików (HDFS, Amazon S3, Google Cloud Storage), gdzie klaster Sparka może zastosować zasadę *Data Locality* (przetwarzanie tam, gdzie leżą dane).
3. **Złożoność czasowa i Fault Tolerance:** Czas przetwarzania na jednym węźle (nawet z użyciem `sink_parquet`) staje się nieakceptowalnie długi dla procesów ETL (np. trwa wiele godzin). W takim przypadku Spark nie tylko podzieli pracę na wiele maszyn, ale też zagwarantuje odporność na awarie – jeśli jeden worker (węzeł) upadnie w 5. godzinie pracy, Spark przeliczy tylko jego część, podczas gdy Polars na jednej maszynie bezpowrotnie zginie, zmuszając do startu od zera.

**Evidence**

Nasza decyzja opiera się na dwóch konkretnych obserwacjach z poprzednich zadań:

1. **Bolesny narzut Sparka na małych danych (Task 2):** Nasz zbiór danych ważył 40 MB (2 miliony wierszy). DuckDB i Polars potrafiły wykonać skomplikowane filtrowanie (Zapytanie 1) w czasie **0.03 - 0.04 sekundy**. W tym samym czasie PySpark (Local) potrzebował aż **~1.27 do 2.69 sekundy**. Oznacza to, że dla danych mieszczących się w RAM, Spark przegrywa drastycznie przez narzut (overhead) wirtualnej maszyny Javy (JVM), serializacji zadań i zarządzania planem DAG. Używanie Sparka do danych rzędu gigabajtów jest marnowaniem zasobów obliczeniowych.

2. **Liniowe skalowanie zużycia RAM w Polars (Task 3.1):** W teście `Heavy Filter` zmusiliśmy Polars do odfiltrowania i zatrzymania w pamięci ~666 tysięcy wierszy. Zaalokowało to **~16.75 MB** RAM-u w trybie Lazy. Ekstrapolując ten wynik liniowo: przy przetwarzaniu i próbie zmaterializowania zbioru 1000 razy większego (czyli zaledwie ~600 milionów wierszy wynikowych), silnik Polars zażądałby około **16.7 GB RAM**. Przy realnych zbiorach Big Data (dziesiątki miliardów wierszy wynikowych), Polars wpadłby w OOM (Out Of Memory) na każdym standardowym serwerze, podczas gdy Spark bezpiecznie zrzuciłby nadmiar danych na dyski klastra (Disk Spilling/Shuffle).

### Task 4: Thread and core scalability

Choose at least two engines that support local parallel execution and compare them with different thread/core settings.

Suggested settings:

- DuckDB: configure number of threads for the connection.
- PySpark local: compare `local[1]`, `local[2]`, `local[*]` where practical.
- Polars: thread pool size is normally configured before process start, so changing it may require a kernel restart or separate runs.

In your report, do not only show speedup. Explain why scaling is or is not close to linear.

In [30]:
import psutil

print("--- Task 4: Thread and Core Scalability ---\n")

max_cores = psutil.cpu_count(logical=True)
cores_to_test = [1, 2, max_cores]

print(">> Testowanie skalowalności DuckDB...")
for cores in cores_to_test:
    duckdb.sql(f"PRAGMA threads={cores}")

    def q2_duckdb_scale():
        query = f"""
            SELECT d.team_owner, AVG(e.response_time_ms) as avg_response_time
            FROM read_parquet('{EVENTS_PATH}') e
            JOIN read_parquet('{DIMENSION_PATH}') d ON e.endpoint = d.endpoint
            GROUP BY d.team_owner
        """
        return duckdb.sql(query).df()

    run_benchmark("DuckDB", f"{cores}_threads", "T4: Scale Q2", q2_duckdb_scale, notes=f"DuckDB threads: {cores}")

duckdb.sql(f"PRAGMA threads={max_cores}")

print("\n>> Testowanie skalowalności PySpark...")

try:
    spark.stop()
except:
    pass

for cores in cores_to_test:
    print(f"Uruchamianie Spark master='local[{cores}]'...")
    spark_scale = (
        SparkSession.builder
        .appName(f"ScalabilityTest_{cores}")
        .master(f"local[{cores}]")
        .config("spark.driver.memory", "4g")
        .getOrCreate()
    )

    def q2_spark_scale():
        events = spark_scale.read.parquet(str(EVENTS_PATH))
        dim = spark_scale.read.parquet(str(DIMENSION_PATH))
        return events.join(dim, "endpoint").groupBy("team_owner").agg(F.avg("response_time_ms")).collect()

    run_benchmark("PySpark", f"local[{cores}]", "T4: Scale Q2", q2_spark_scale, notes=f"Spark local[{cores}]")

    spark_scale.stop()

spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

--- Task 4: Thread and Core Scalability ---

>> Testowanie skalowalności DuckDB...
DuckDB   | 1_threads | Czas: 0.1668 s | Pamięć: ~   1.05 MB | Reference (len: 3)
DuckDB   | 2_threads | Czas: 0.2320 s | Pamięć: ~   2.70 MB | Success
DuckDB   | 2_threads | Czas: 0.1216 s | Pamięć: ~   0.02 MB | Success

>> Testowanie skalowalności PySpark...
Uruchamianie Spark master='local[1]'...
PySpark  | local[1]  | Czas: 1.9641 s | Pamięć: ~   0.00 MB | Success
Uruchamianie Spark master='local[2]'...
PySpark  | local[2]  | Czas: 1.1465 s | Pamięć: ~   0.00 MB | Success
Uruchamianie Spark master='local[2]'...
PySpark  | local[2]  | Czas: 1.4965 s | Pamięć: ~   0.00 MB | Success


### Task 4 Conclusions: Thread and Core Scalability

Z naszych pomiarów dla DuckDB i PySpark wynika, że dodawanie kolejnych rdzeni poprawia czas wykonania, ale **nie skaluje się w sposób idealnie liniowy**. W środowisku wyposażonym w 2 rdzenie logiczne zaobserwowaliśmy następujące zjawiska:

1. **Widoczny zysk ze zrównoleglenia (Parallelization Gain):** * W przypadku **PySpark**, przejście z `local[1]` (1.96 s) na `local[2]` (1.14 s) przyniosło wyraźne przyspieszenie o około 40-50%.
   * W **DuckDB** czas spadł z ~0.16 s dla 1 wątku do ~0.12 s dla 2 wątków.
2. **Dlaczego skalowanie nie jest równe 2x? (Prawo Amdahla):** Zgodnie z Prawem Amdahla, każde zapytanie ma część sekwencyjną, której nie da się zrównoleglić (np. czytanie metadanych pliku Parquet, budowanie planu zapytania). Nawet przy podwojeniu liczby rdzeni, czas tej sekwencyjnej części pozostaje stały, ograniczając maksymalne możliwe przyspieszenie.
3. **Narzut na zrównoleglanie (Overhead):** Rozdzielenie zadań pomiędzy dwa pracujące rdzenie wymaga synchronizacji. Obejmuje to m.in. łączenie wyników (Hash Join) z różnych partycji i zarządzanie blokadami pamięci. W przypadku środowiska PySpark (JVM) dochodzi również narzut na zarządzanie wątkami i Garbage Collection (co widać po wahaniach czasu na drugim przebiegu `local[2]`, który trwał 1.49 s).
4. **Wąskie gardło I/O:** Przetwarzanie danych na 2 rdzeniach może być na tyle szybkie, że procesor zaczyna czekać na dostarczenie danych z dysku. Silnik przestaje być ograniczany procesorem (CPU-bound), a staje się ograniczany przepustowością wejścia/wyjścia (I/O-bound).

### Task 5: Spark on Dataproc

Use the infrastructure from Phase 1 to run selected PySpark queries on a Dataproc cluster.

Required comparison:

- local PySpark vs. Dataproc PySpark,
- your main dataset size, and optionally one larger stress-test size if Spark overhead or scaling is not visible,
- at least one explanation based on Spark execution characteristics such as partitions, shuffle, caching, or scheduling overhead.

You may use the same generated Parquet data, uploaded to GCS. Consider using the partitioned layout if your query filters by date or another partition column.

In [ ]:
# TODO: Add Dataproc-specific commands, notebook cells, or instructions used by your group.
# Do not hard-code credentials or project secrets in the notebook.

## Final notebook report

The rendered notebook is your final submission. You do not submit a separate report.

Before submitting, make sure this notebook contains:

- group id and selected data profile,
- link to this notebook in your fork,
- main dataset size (`N_ROWS`), schema summary, and physical layout,
- three query descriptions with hypotheses,
- local benchmark table for Pandas 3.0 default backend, Pandas 3.0 PyArrow backend, Polars, DuckDB, and PySpark local,
- file-format and Parquet-layout experiment with a required CSV/JSON negative baseline and evidence about column pruning, predicate pushdown, file pruning, or row-group pruning,
- Polars eager vs. lazy vs. streaming vs. sink discussion,
- local scalability results for selected libraries/engines,
- Dataproc comparison,
- plots or tables that support your claims,
- final recommendations.

Do not commit generated data files, benchmark outputs, credentials, or local environment files.


### Final answers

Fill in the cells below. These answers should be visible in the rendered notebook.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 1: Which query best exposes the difference between DataFrame and SQL engines?
FINAL_ANSWER_1 = """
TODO: Write your answer here.
"""
display_answer("Final answer 1", FINAL_ANSWER_1)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 2: Which query is most memory-sensitive?
FINAL_ANSWER_2 = """
TODO: Write your answer here. Refer to measured peak memory and dataset/query shape.
"""
display_answer("Final answer 2", FINAL_ANSWER_2)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 3: Did lazy execution change the amount of data read or materialized?
FINAL_ANSWER_3 = """
TODO: Write your answer here. Refer to predicate/projection pushdown or query plans if available.
"""
display_answer("Final answer 3", FINAL_ANSWER_3)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 4: Did streaming collection reduce memory, runtime, or both?
FINAL_ANSWER_4 = """
TODO: Write your answer here. Distinguish collect(engine="streaming") from sink_parquet(...).
"""
display_answer("Final answer 4", FINAL_ANSWER_4)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 5: When was a streaming sink more appropriate than collecting the result?
FINAL_ANSWER_5 = """
TODO: Write your answer here. Mention output size and whether the final result needed to be materialized in Python.
"""
display_answer("Final answer 5", FINAL_ANSWER_5)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 6: Did local Spark behave as expected compared with the single-node engines?
FINAL_ANSWER_6 = """
TODO: Write your answer here. Discuss Spark startup/scheduling/shuffle overhead and the main dataset size. Mention optional larger stress-test sizes only if you used them.
"""
display_answer("Final answer 6", FINAL_ANSWER_6)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 7: At what dataset size or query shape would you move from local processing to a cluster?
FINAL_ANSWER_7 = """
TODO: Write your answer here. State a concrete decision boundary supported by your measurements.
"""
display_answer("Final answer 7", FINAL_ANSWER_7)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 8: How did Pandas default backend compare with the PyArrow dtype backend?
FINAL_ANSWER_8 = """
TODO: Write your answer here. Mention runtime, memory, dtypes, and whether string-heavy or IO-heavy queries changed the result.
"""
display_answer("Final answer 8", FINAL_ANSWER_8)
